# Latent class modeling

Use StepMix to fit a latent class model, so a mixture of categorical or continuous variables, using the EM-algorithm.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import rand_score
from stepmix.stepmix import StepMix
from stepmix.utils import get_mixed_descriptor

from utils import plot_corr

# Hardcoded RNG object for this script for reproducibility
RANDOM_SEED = 1270
rng = np.random.default_rng(RANDOM_SEED)


# Hardcoded paths, change to apply to new data
data_path = "data/preprocessed/esa_2018.csv"
vars_info_path = "data/preprocessed/esa_grouped_vars_info.csv"
outputs_folder = "outputs"
allow_overwrite = True

In [ ]:
# Load data and information about variables
data = pd.read_csv(data_path, index_col=0, low_memory=False)
vars_info = pd.read_csv(vars_info_path, index_col=0, low_memory=False)

# Define variables
indicator_vars = (
    pd.Series(vars_info.index[vars_info["type"] == "categorical"])  # change which variables are included later
    .sample(n=40, random_state=np.random.RandomState(rng.bit_generator))  # reduce size for now
    .values
)

training_data = (
    data.loc[:, indicator_vars].sample(
        n=1000, axis=0, random_state=np.random.RandomState(rng.bit_generator)
    )  # reduce size for now
)
num_respondents, num_questions = training_data.shape  # N, J
num_choices = [x.max() + 1 for var, x in training_data.items()]  # A

# Plot the initial correlation matrix
plot_corr(training_data, var_names=indicator_vars, tick_label_size=9)
plt.show()

## Specify and fit the model

The class probabilities are parameters\
($\gamma_c$ is the prob. of class $c$):\
$\gamma = (\gamma_0, \gamma_1, \dotsc, \gamma_{K - 1})$ $\qquad$ constrained by: $\;$ $\displaystyle\sum_{c = 0}^{K - 1} \gamma_c = 1$

The response probabilities are also parameters\
($\pi(c)_{j a}$ is the prob. of a person in class $c$ to answer to the $j$-th question with the $a$-th answer option):\
$\forall c \in \{0, 1, \dotsc, K - 1\}, j \in \{0, 1, \dotsc, A_j - 1\}$\
$\pi(c)_j = (\pi(c)_{j 0}, \pi(c)_{j 1}, \dotsc, \pi(c)_{j A_j - 1})$ $\qquad$ constrained by: $\;$ $\displaystyle\sum_{a = 0}^{A_j - 1} \pi(c)_{j a} = 1$

A person's latent class is a categorical random variable:\
$C \sim \mathrm{Cat}(\mathbf{\gamma})$

A person's answers to the $j$-th and ${j'}$-th questions, $X_j$ and $X_{j'}$, are independent conditional on their class membership:\
$j \neq j' \; \Longrightarrow \; X_j \perp\mkern -10mu\perp X_{j'} \mid C = c$\
The answers $X = (X_0, X_1, \dotsc, X_{J - 1})$ are thus taken as indicator variables for the latent factor $C$.

The conditional distributions of a person's answers are called the measurement model (may be continuous for an answer):\
$X_j \mid C = c \sim \mathrm{Cat}(\pi(c)_j)$ $\quad$ $\big($ or $\quad$ $X_j \mid C = c \sim \text{continuous}$ $\big)$

In other words: $\quad$ $X_j$ follows a (discrete or continuous) finite mixture of random variables, with mixture weights $\gamma$.\
$\displaystyle P(X_j = a) = \sum_{c = 0}^{K - 1} \gamma_c \, P(X_j = a \mid C = c)$

The likelihood corresponding to one person is:\
$\displaystyle P(X = a) = P(X_0 = a_0, X_1 = a_1, \dotsc, X_{J - 1} = a_{J - 1}) = \sum_{c = 0}^{K - 1} \gamma_c \prod_{j = 0}^{J - 1} P(X_j = a_j \mid C = c)$

Introducing a population indexed by $i \in \{0, 1, \dotsc, N - 1\}$, the complete likelihood is:\
$P(X = a) = P(X_0 = a_0, X_1 = a_1, \dotsc, X_{N - 1} = a_{N - 1})$\
$\displaystyle= P(X_{0 0} = a_{0 0}, X_{0 1} = a_{0 1}, \dotsc, X_{N - 1 J - 2} = a_{N - 1 J - 2}, X_{N - 1 J - 1} = a_{N - 1 J - 1})$\
$\displaystyle= \prod_{i = 0}^{N - 1} \sum_{c_i = 0}^{K - 1} \gamma_{c_i} \prod_{j = 0}^{J - 1} P(X_{i j} = a_{i j} \mid C_i = c_i)$

In [ ]:
# Choose categorical (and possibly continuous) indicators for the latent classes
num_latent = 10  # K
continuous_vars = []  # empty for now
categorical_vars = [var for var in indicator_vars if var not in continuous_vars]

training_data, measurement_vars = get_mixed_descriptor(
    dataframe=training_data,
    # continuous=continuous_vars,  # empty for now
    categorical=categorical_vars,
)
model = StepMix(
    n_components=num_latent,  # specify number of latent classes
    n_steps=1,  # select normal EM-algorithm
    measurement=measurement_vars,  # specify the measurement model (conditional answer distributions)
    random_state=np.random.RandomState(rng.bit_generator),
    verbose=1,
)

In [ ]:
model.fit(training_data)

In [ ]:
print("Model converged." if model.converged_ else "Model did not converge.")

## Model goodness-of-fit measures

In [ ]:
bic = model.bic(training_data)
sabic = model.sabic(training_data)
aic = model.aic(training_data)
caic = model.caic(training_data)
entropy = model.entropy(training_data)
relative_entropy = model.relative_entropy(training_data)
print(
    f"BIC = {bic.item()}, SABIC = {sabic.item()},\n"
    f"AIC = {aic.item()}, CAIC = {caic.item()},\n"
    f"entropy = {entropy.item()}, relative entropy = {relative_entropy.item()}"
)

## Model information

In [ ]:
# Parameter count
model.n_parameters

In [ ]:
# Factor loadings
model.get_mm_df()

In [ ]:
# Class weights
model.get_cw_df()

## Predictions

In [ ]:
# Predict class membership
data["pred_cls"] = model.predict(data[indicator_vars])

In [ ]:
# Predict class membership probabilities
model.predict_proba(training_data)

In [ ]:
# Sample from the modeled data generating process
print(model.sample(1))

In [ ]:
# Plot correlation matrix conditional on class membership for every class, to show conditional independence
# ...

## Bootstrap estimation

In [ ]:
model.bootstrap(training_data, n_repetitions=10)
# model.bootstrap_stats(training_data, n_repetitions=10)

## Cluster validation

The next step is to fit another model (same specification) on the 2021 dataset, and compare the predictions of the 2018 model and the 2021 model on the 2021 dataset (e.g. using the Rand score).